# Semester 4 — Multi-Agent Architecture: Práctica

**Objetivo:** Implementar los patrones fundamentales de sistemas multi-agente: orquestación, dependency graphs, paralelismo, estado compartido y circuit breaker.

Cada patrón se implementa mínimamente en Python puro para que la mecánica sea transparent. Frameworks como LangGraph añaden capas encima de esto — conocer la base te permite evaluar cuándo esas capas añaden valor real.

**Prerequisito:** Variable de entorno `OPENAI_API_KEY` configurada.

## Setup

In [ ]:
!pip install openai --quiet

## Ejercicio 1 — Orquestador que delega a agentes especializados

**Qué practicamos:** Implementar un orquestador que clasifica la tarea entrante y la delega al agente especializado correcto.

**Por qué importa arquitectónicamente:** La orquestación es el patrón de composición más importante en sistemas multi-agente. El orquestador no sabe cómo hacer las tareas — sabe a quién delegarlas. Esta separación permite añadir agentes especializados sin modificar el orquestador, y escalar cada agente de forma independiente.

In [ ]:
import os
import json
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# Agentes especializados
def code_agent(task: str) -> str:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Eres un experto en programación. Responde solo con código Python limpio y un comentario explicativo mínimo."},
            {"role": "user",   "content": task}
        ],
        temperature=0
    )
    return r.choices[0].message.content

def analysis_agent(task: str) -> str:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Eres un analista de datos. Proporciona análisis estructurados con conclusiones claras y accionables."},
            {"role": "user",   "content": task}
        ],
        temperature=0
    )
    return r.choices[0].message.content

def writing_agent(task: str) -> str:
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "Eres un redactor técnico. Escribe de forma clara, concisa y profesional."},
            {"role": "user",   "content": task}
        ],
        temperature=0.3
    )
    return r.choices[0].message.content

AGENTS = {
    "code":     code_agent,
    "analysis": analysis_agent,
    "writing":  writing_agent,
}

def orchestrator(task: str) -> str:
    """Clasifica la tarea y delega al agente correcto."""
    classification_prompt = f"""Clasifica esta tarea en una de: code, analysis, writing.
Responde SOLO con JSON: {{"agent": "code|analysis|writing", "reason": "<una frase>"}}

Tarea: {task}"""

    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": classification_prompt}],
        response_format={"type": "json_object"},
        temperature=0
    )
    routing = json.loads(r.choices[0].message.content)
    agent_name = routing["agent"]

    print(f"  → Orquestador: delegando a '{agent_name}' ({routing['reason']})")

    agent_fn = AGENTS.get(agent_name, writing_agent)
    return agent_fn(task)

tasks = [
    "Escribe una función Python que calcule el percentil 95 de una lista de números",
    "Analiza las tendencias de una empresa con revenue de $1M en Q1, $1.4M en Q2, $1.2M en Q3",
    "Redacta un email para clientes anunciando una nueva funcionalidad de IA en el producto",
]

for task in tasks:
    print(f"\nTarea: '{task[:60]}...'" if len(task) > 60 else f"\nTarea: '{task}'")
    result = orchestrator(task)
    print(f"  Resultado: {result[:200]}..." if len(result) > 200 else f"  Resultado: {result}")

## Ejercicio 2 — Dependency Graph: ejecutar tareas en orden topológico

**Qué practicamos:** Representar tareas con dependencias como un DAG y ejecutarlas en el orden correcto usando ordenación topológica.

**Por qué importa arquitectónicamente:** Los sistemas multi-agente reales tienen dependencias entre tareas. El agente B necesita el output del agente A. Sin gestión de dependencias, ejecutas las tareas en orden arbitrario y obtienes resultados incorrectos o crashs. El grafo de dependencias es la estructura de datos que hace posible la orquestación robusta.

In [ ]:
import os
from collections import deque
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

def topological_sort(tasks: dict) -> list[str]:
    """Kahn's algorithm. tasks = {id: {deps: [ids], prompt_template: str}}"""
    in_degree = {tid: len(t["deps"]) for tid, t in tasks.items()}
    queue = deque(tid for tid, deg in in_degree.items() if deg == 0)
    order = []

    while queue:
        tid = queue.popleft()
        order.append(tid)
        for other_id, other_task in tasks.items():
            if tid in other_task["deps"]:
                in_degree[other_id] -= 1
                if in_degree[other_id] == 0:
                    queue.append(other_id)

    if len(order) != len(tasks):
        raise ValueError("Ciclo detectado en el grafo de dependencias")
    return order

def run_task(task_id: str, task: dict, outputs: dict) -> str:
    """Ejecuta una tarea usando los outputs de sus dependencias."""
    prompt = task["prompt"]
    for dep_id in task["deps"]:
        prompt += f"\n\n[Output de {dep_id}]: {outputs[dep_id]}"

    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        max_tokens=200
    )
    return r.choices[0].message.content

# Pipeline de análisis con dependencias:
# research → analysis → summary → email
#                ↑
#           data_extract
PIPELINE = {
    "research":     {"deps": [],                  "prompt": "Lista 3 tendencias actuales en adopción empresarial de IA en 2024."},
    "data_extract": {"deps": [],                  "prompt": "Proporciona 3 métricas clave sobre el mercado de IA: tamaño, crecimiento y adopción."},
    "analysis":     {"deps": ["research", "data_extract"], "prompt": "Combina las tendencias y métricas anteriores en un análisis ejecutivo de 3 puntos."},
    "summary":      {"deps": ["analysis"],         "prompt": "Resume el análisis anterior en 2 frases para un CTO."},
    "email":        {"deps": ["summary"],          "prompt": "Redacta un email ejecutivo de 100 palabras basado en el resumen para presentarlo a la junta."},
}

order = topological_sort(PIPELINE)
print(f"Orden de ejecución: {' → '.join(order)}\n")

outputs = {}
for task_id in order:
    print(f"Ejecutando: {task_id}")
    outputs[task_id] = run_task(task_id, PIPELINE[task_id], outputs)
    print(f"  → {outputs[task_id][:100]}...\n" if len(outputs[task_id]) > 100 else f"  → {outputs[task_id]}\n")

print("=== EMAIL FINAL ===")
print(outputs["email"])

## Ejercicio 3 — Ejecución paralela con asyncio.gather

**Qué practicamos:** Lanzar múltiples llamadas LLM en paralelo usando `asyncio.gather` y medir la diferencia de latencia vs. ejecución secuencial.

**Por qué importa arquitectónicamente:** Las tareas independientes en un pipeline multi-agente no deben ejecutarse secuencialmente. El paralelismo reduce la latencia total de O(n × latencia_LLM) a O(max_latencia_LLM). En sistemas de producción esto puede ser la diferencia entre 10s y 2s de tiempo de respuesta.

In [ ]:
import os
import asyncio
import time
from openai import AsyncOpenAI

async_client = AsyncOpenAI(api_key=os.environ["OPENAI_API_KEY"])

TASKS = [
    {"id": "sentiment", "prompt": "Analiza el sentimiento de: 'El producto es increíble pero el soporte es pésimo'. Responde en una línea."},
    {"id": "category",  "prompt": "Clasifica en una palabra: 'El producto es increíble pero el soporte es pésimo'. Categorías: positivo/negativo/mixto"},
    {"id": "action",    "prompt": "Propón una acción concreta para: 'El producto es increíble pero el soporte es pésimo'. Una frase."},
    {"id": "priority",  "prompt": "Prioridad (alta/media/baja) para: 'El producto es increíble pero el soporte es pésimo'. Responde en una palabra."},
]

async def run_single(task: dict) -> tuple[str, str]:
    r = await async_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": task["prompt"]}],
        temperature=0,
        max_tokens=50
    )
    return task["id"], r.choices[0].message.content.strip()

async def run_parallel() -> dict:
    results = await asyncio.gather(*[run_single(t) for t in TASKS])
    return dict(results)

# Secuencial para comparar
from openai import OpenAI
sync_client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

print("=== Ejecución SECUENCIAL ===")
t0 = time.time()
sequential_results = {}
for task in TASKS:
    r = sync_client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": task["prompt"]}],
        temperature=0, max_tokens=50
    )
    sequential_results[task["id"]] = r.choices[0].message.content.strip()
t_seq = time.time() - t0
print(f"Tiempo: {t_seq:.2f}s")
for k, v in sequential_results.items():
    print(f"  {k}: {v}")

print("\n=== Ejecución PARALELA ===")
t0 = time.time()
parallel_results = asyncio.run(run_parallel())
t_par = time.time() - t0
print(f"Tiempo: {t_par:.2f}s")
for k, v in parallel_results.items():
    print(f"  {k}: {v}")

print(f"\nSpeedup: {t_seq / t_par:.1f}x más rápido en paralelo")

## Ejercicio 4 — Estado compartido con threading.Lock

**Qué practicamos:** Implementar estado compartido entre agentes que se ejecutan en paralelo, usando `threading.Lock` para evitar condiciones de carrera.

**Por qué importa arquitectónicamente:** En sistemas multi-agente paralelos, múltiples agentes pueden necesitar leer y escribir estado compartido (contador de llamadas, presupuesto de tokens, memoria de conversación). Sin sincronización, obtienes race conditions que producen resultados incorrectos de forma no reproducible — el peor tipo de bug en producción.

In [ ]:
import os
import threading
import concurrent.futures
import time
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

# ponytail: Lock global por simplicidad. En producción: lock por recurso específico (por usuario, por presupuesto).
class SharedBudget:
    def __init__(self, max_tokens: int):
        self.remaining = max_tokens
        self.spent = 0
        self.calls = 0
        self._lock = threading.Lock()

    def consume(self, tokens: int, agent_id: str) -> bool:
        """Intenta consumir tokens. Retorna True si hay presupuesto, False si no."""
        with self._lock:
            if self.remaining >= tokens:
                self.remaining -= tokens
                self.spent += tokens
                self.calls += 1
                print(f"  [{agent_id}] consumió {tokens} tokens | restante: {self.remaining}")
                return True
            print(f"  [{agent_id}] BLOQUEADO: solo {self.remaining} tokens restantes, necesita {tokens}")
            return False

    def stats(self) -> dict:
        with self._lock:
            return {"spent": self.spent, "remaining": self.remaining, "calls": self.calls}

def agent_task(agent_id: str, task: str, budget: SharedBudget) -> str:
    """Agente que verifica presupuesto antes de llamar al LLM."""
    ESTIMATED_TOKENS = 100  # estimación pre-llamada

    if not budget.consume(ESTIMATED_TOKENS, agent_id):
        return f"[{agent_id}] Omitido: presupuesto insuficiente"

    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": task}],
        temperature=0,
        max_tokens=50
    )
    return f"[{agent_id}] {r.choices[0].message.content.strip()[:80]}"

# 5 agentes compiten por un presupuesto de 350 tokens
budget = SharedBudget(max_tokens=350)

agent_tasks = [
    ("agent-A", "Nombra una ventaja de RAG. Una frase."),
    ("agent-B", "Nombra una desventaja de RAG. Una frase."),
    ("agent-C", "Nombra una alternativa a RAG. Una frase."),
    ("agent-D", "¿Cuándo no usar RAG? Una frase."),
    ("agent-E", "¿Qué modelo usar para embeddings? Una frase."),
]

print(f"Presupuesto inicial: {budget.remaining} tokens\n")

with concurrent.futures.ThreadPoolExecutor(max_workers=3) as executor:
    futures = [executor.submit(agent_task, aid, task, budget) for aid, task in agent_tasks]
    results = [f.result() for f in concurrent.futures.as_completed(futures)]

print("\n=== Resultados ===")
for r in results:
    print(r)

print(f"\n=== Stats de presupuesto ===")
print(budget.stats())

## Ejercicio 5 — Circuit Breaker: fallback automático tras fallos consecutivos

**Qué practicamos:** Implementar un circuit breaker que deja de llamar a un agente si falla N veces consecutivas, usando un fallback en su lugar.

**Por qué importa arquitectónicamente:** En sistemas distribuidos, un servicio degradado puede causar que todos los requests esperen el timeout antes de fallar. El circuit breaker detecta el fallo rápido, evita cascadas de errores, y permite recuperación gradual. Es el patrón de resiliencia más importante en cualquier sistema de producción con dependencias externas.

In [ ]:
import os
import time
import random
from enum import Enum
from openai import OpenAI

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])

class CircuitState(Enum):
    CLOSED = "closed"      # normal, todo bien
    OPEN   = "open"        # disparado, usando fallback
    HALF_OPEN = "half_open"  # probando recuperación

class CircuitBreaker:
    def __init__(self, failure_threshold: int = 3, recovery_timeout: float = 5.0):
        self.state = CircuitState.CLOSED
        self.failure_threshold = failure_threshold
        self.consecutive_failures = 0
        self.last_failure_time = 0.0
        self.recovery_timeout = recovery_timeout

    def call(self, fn, *args, fallback=None, **kwargs):
        if self.state == CircuitState.OPEN:
            if time.time() - self.last_failure_time > self.recovery_timeout:
                print(f"  [CIRCUIT] → HALF_OPEN: probando recuperación")
                self.state = CircuitState.HALF_OPEN
            else:
                print(f"  [CIRCUIT] OPEN: usando fallback")
                return fallback(*args, **kwargs) if fallback else None

        try:
            result = fn(*args, **kwargs)
            if self.state == CircuitState.HALF_OPEN:
                print(f"  [CIRCUIT] → CLOSED: recuperado")
            self.state = CircuitState.CLOSED
            self.consecutive_failures = 0
            return result
        except Exception as e:
            self.consecutive_failures += 1
            self.last_failure_time = time.time()
            print(f"  [CIRCUIT] Fallo #{self.consecutive_failures}: {e}")
            if self.consecutive_failures >= self.failure_threshold:
                print(f"  [CIRCUIT] → OPEN: umbral de fallos alcanzado")
                self.state = CircuitState.OPEN
            if fallback:
                return fallback(*args, **kwargs)
            raise

# Agente que falla aleatoriamente (simula servicio inestable)
FAILURE_RATE = 0.6

def unreliable_agent(task: str) -> str:
    if random.random() < FAILURE_RATE:
        raise RuntimeError("Servicio temporalmente no disponible")
    r = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": task}],
        temperature=0, max_tokens=30
    )
    return r.choices[0].message.content.strip()

def fallback_agent(task: str) -> str:
    """Respuesta simplificada cuando el agente principal no está disponible."""
    return "[FALLBACK] Servicio no disponible. Por favor reintenta en unos momentos."

cb = CircuitBreaker(failure_threshold=3, recovery_timeout=2.0)
random.seed(42)  # reproducibilidad en el demo

print("Enviando 8 requests al agente inestable...\n")
for i in range(8):
    task = f"Responde en 5 palabras: ¿qué es un agente de IA? (request #{i+1})"
    result = cb.call(unreliable_agent, task, fallback=fallback_agent)
    print(f"Request {i+1} [{cb.state.value:9s}]: {result}")
    time.sleep(0.3)